# SmartCrop — Notebook 02: Data Cleaning

**Study area:** Macheke, Mashonaland East, Zimbabwe  
**Crop:** maize only  
**Target:** `yield_t_ha` (tonnes per hectare)  
**Input:** `data/interim/consolidated_raw.csv`  
**Output:** `data/processed/clean_dataset.csv`

> The input file is synthetic. Cleaning does not turn it into a Macheke survey.


## 1. Introduction

Notebook 01 consolidated two synthetic batches and left duplicates, blank cells, and a few impossible values in place. This notebook applies a written cleaning policy and saves the result. It does not impute feature gaps. Imputation is fitted later, on the training seasons only, so the clean file does not leak the test seasons.


## 2. Objectives

1. Count rows, missing cells, and exact duplicates before cleaning.
2. Remove exact duplicate rows.
3. Blank values outside physical bounds and outside the documented plausible bounds.
4. Blank average temperature where `min <= avg <= max` fails.
5. Drop rows with no yield. Those rows cannot supervise a model.
6. Export before/after tables as CSV, Excel, and HTML.


## 3. Dataset Description

The table is one synthetic farm in one synthetic season. `data_source_type` remains `SYNTHETIC`. Ward codes remain placeholders. Season labels remain labels, not a historical weather series. Yield is tonnes per hectare.


## 4. Methodology

Physical bounds mark values that cannot be a measurement of that quantity, such as a negative rainfall. Plausible bounds are project checks for the injected extremes, not official agronomic limits. A value outside either bound becomes missing. The rest of the row stays.

Temperature order is checked only when minimum, average, and maximum are all present. If the order fails, the average is blanked because that is the field the generator corrupted.

Exact duplicates are identical on every column, including `record_id`. The first copy is kept.

Rows with missing `yield_t_ha` are dropped after the blanking steps, so a yield that was an injected extreme is also dropped. Feature missingness is left for the model pipeline.


## 5. Implementation


In [10]:
from pathlib import Path
import sys
import pandas as pd

def find_root() -> Path:
    start = Path.cwd().resolve()
    candidates = [start, *start.parents]
    if (start / "data-science").is_dir():
        candidates.insert(0, start / "data-science")
    for candidate in candidates:
        if (candidate / "src" / "data" / "schema.py").is_file():
            return candidate
    raise FileNotFoundError("Open this notebook from the SmartCrop repository.")

ROOT = find_root()
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 120)

from data.cleaning import clean_observations, quality_tables, write_clean_outputs
from data.schema import coerce_observation_frame

raw = coerce_observation_frame(pd.read_csv(ROOT / "data" / "interim" / "consolidated_raw.csv"))
print(f"Rows before cleaning: {len(raw)}")
print(f"Exact duplicate rows before cleaning: {int(raw.duplicated().sum())}")
print(f"Missing yield rows before cleaning: {int(raw['yield_t_ha'].isna().sum())}")


Rows before cleaning: 974
Exact duplicate rows before cleaning: 14
Missing yield rows before cleaning: 20


In [11]:
clean, report = clean_observations(raw)
tables = quality_tables(raw, clean)
paths = write_clean_outputs(clean, report, tables, ROOT)
print(f"Rows after cleaning: {report['rows_after']}")
print(f"Exact duplicates removed: {report['exact_duplicates_removed']}")
print(f"Temperature rows repaired: {report['temperature_order_repairs']}")
print(f"Rows dropped for missing yield: {report['rows_dropped_for_missing_yield']}")
print(f"Missing cells before: {report['missing_cells_before']}")
print(f"Missing cells after: {report['missing_cells_after']}")
print(f"Imputation: {report['imputation']}")
display(tables["before_after"])
display(tables["missing_by_column"])


Rows after cleaning: 938
Exact duplicates removed: 14
Temperature rows repaired: 3
Rows dropped for missing yield: 22
Missing cells before: 320
Missing cells after: 308
Imputation: none in the clean file; the model pipeline imputes training medians


,stage,rows,columns,missing_cells,duplicate_rows
0,before,974,23,320,14
1,after,938,23,308,0


,column,missing_before,missing_after
0,record_id,0,0
1,farm_id,0,0
2,year,0,0
3,season,0,0
4,country,0,0
5,province,0,0
6,location,0,0
7,ward,0,0
8,crop,0,0
9,area_ha,27,25


## 6. Results

The tables above are the cleaning log. Duplicate rows should be zero after this step. Yield should have no missing values. Some feature cells remain missing on purpose.


## 7. Visualizations

Cleaning is summarised as a table rather than a chart. The missing-value figure for the cleaned file is drawn in Notebook 04, after this file exists.


## 8. Interpretation

Removing exact copies stops the same farm-season from counting twice. Blanking impossible and extreme values keeps the farm row but stops a single corrupted cell from dominating a split. Dropping missing yields is required for supervised learning. Leaving other gaps unfilled avoids choosing a median from seasons that will later be used as the test set.


## 9. Conclusion

`data/processed/clean_dataset.csv` is the modelling table. It is still synthetic maize data for the Macheke study area. The next notebook adds a short list of features that a prediction request can recompute.
